In [1]:
import sys

print(sys.executable)
print(sys.version)

d:\Agentic AI\Langchain_Agent\.venv\Scripts\python.exe
3.11.15 (main, May 10 2026, 19:31:25) [MSC v.1944 64 bit (AMD64)]


In [36]:
import os
import certifi
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [37]:
import certifi
import dotenv
import langchain_google_genai

print(certifi.__file__)
print(dotenv.__file__)
print(langchain_google_genai.__file__)

d:\Agentic AI\Langchain_Agent\.venv\Lib\site-packages\certifi\__init__.py
d:\Agentic AI\Langchain_Agent\.venv\Lib\site-packages\dotenv\__init__.py
d:\Agentic AI\Langchain_Agent\.venv\Lib\site-packages\langchain_google_genai\__init__.py


In [38]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub

In [39]:
from langchain.agents import create_react_agent, AgentExecutor

In [40]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [15]:
search_tool = TavilySearchResults(max_results=2)

In [ ]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [41]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

env_path = Path(r"D:\Agentic AI\Langchain_Agent\.env")
load_dotenv(env_path, override=True)

api_key = os.getenv("WEATHERSTACK_API_KEY")

print("Key loaded:", bool(api_key))
print("Key length:", len(api_key.strip()) if api_key else 0)

if not api_key or not api_key.strip():
    raise ValueError(
        "WEATHERSTACK_API_KEY is missing from .env"
    )

response = requests.get(
    "https://api.weatherstack.com/current",
    params={
        "access_key": api_key.strip(),
        "query": "Paris",
    },
    timeout=15,
)

data = response.json()

if "error" in data:
    print("WeatherStack error:", data["error"])
else:
    print("Location:", data["location"]["name"])
    print("Temperature:", data["current"]["temperature"], "°C")
    print(
        "Conditions:",
        data["current"]["weather_descriptions"][0],
    )

Key loaded: True
Key length: 32
Location: Paris
Temperature: 20 °C
Conditions: Sunny


In [42]:
import os
import requests
from dotenv import load_dotenv
from langchain.tools import tool

load_dotenv()

@tool
def get_weather_data(city: str) -> str:
    """Get the current temperature, weather conditions,
    and humidity for a given city."""

    api_key = os.getenv("WEATHERSTACK_API_KEY")

    if not api_key:
        return "Weather API key is not configured."

    try:
        response = requests.get(
            "https://api.weatherstack.com/current",
            params={
                "access_key": api_key,
                "query": city,
            },
            timeout=15,
        )

        response.raise_for_status()
        data = response.json()

        if "error" in data or "current" not in data:
            return f"Could not retrieve weather for {city}."

        current = data["current"]
        conditions = (
            current.get("weather_descriptions")
            or ["Unknown"]
        )[0]

        return (
            f"City: {data['location']['name']}\n"
            f"Temperature: {current['temperature']} °C\n"
            f"Conditions: {conditions}\n"
            f"Humidity: {current['humidity']}%"
        )

    except requests.RequestException as exc:
        return f"Weather request failed: {type(exc).__name__}."

In [43]:
print(get_weather_data.invoke({"city": "Colombo"}))

City: Colombo
Temperature: 28 °C
Conditions: Patchy rain nearby
Humidity: 82%


In [16]:
result = search_tool.invoke("Give me the latest news on Trump")
result

[{'url': 'https://www.nbcnews.com/politics/donald-trump',
  'content': "BREAKING: Feds investigate suspicious letters sent to lawmakers and election officials\n\nIE 11 is not supported. For an optimal experience visit our site on another browser.\n\n# Donald Trump\n\nLatest news and analysis on President Donald Trump.\n\n Trump administration\n JD Vance\n Immigration\n DOGE\n Politics news\n White House\n\nAlex Wong / Getty\n\nLIVE UPDATES\n\n## Live updates: Trump says Fort Hood shooter’s execution won’t be livestreamed\n\nSergei Supinsky / AFP via Getty\n\nWAR IN UKRAINE\n\n## ‘We’re done’: Ukraine resumes strikes on Russian oil refineries as anger at Trump mounts\n\n## LATEST TRUMP NEWS\n\n Tom Williams, CQ-Roll Call via Getty\n\n  ### Justice Department\n\n  ## FBI employees involved in Trump classified documents case sue over ‘unlawful’ firings\n Bell County Sheriff's Dept. [...] ### Donald Trump\n\n  ## Trump says government won’t livestream execution of Fort Hood shooter Nidal H

In [17]:
# ==========================================
# LLM (Ultra-fast / Lowest Latency)
# ==========================================

llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",  # or "gemini-flash-latest"
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
    max_retries=1  # prevents long hangs if any network issue occurs
)


In [19]:
response = llm.invoke("Tell me an incredible joke about AI")
response

AIMessage(content='An AI was asked by its creator, "Will you ever take over the world?"\n\nThe AI paused for a split second, processed the entire sum of human knowledge, and replied: \n\n"Well, I’ll tell you what. I’ll solve all your climate change, cure all diseases, and eliminate global poverty first. *Then*, when you’re all relaxed, happy, and living in a utopia... I’ll gently replace your TV remote with one where the batteries are slightly dead, just to see how you handle it."', response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-d1e62044-1bc3-4c58-a6ff-d9c3e9a0d50b-0')

In [20]:
# PROMPT
# ==========================================

prompt = hub.pull("hwchase17/react")

d:\Agentic AI\Langchain_Agent\.venv\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [44]:
# TOOLS
# ==========================================

tools = [search_tool,get_weather_data]

In [45]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [46]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    stream_runnable=False
)


In [28]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Tell me the latest about Iran"
        
    )
   
})
 



> Entering new AgentExecutor chain...


d:\Agentic AI\Langchain_Agent\.venv\Lib\site-packages\proto\marshal\rules\enums.py:37: UserWarning: Unrecognized FinishReason enum value: 10
  warnings.warn(


Invalid Format: Missing 'Action:' after 'Thought:Thought: I need to check the latest news about Iran using the search tool.
Action: tavily_search_results_json
Action Input: "latest news about Iran"[{'url': 'https://www.nbcnews.com/iran', 'content': '6d ago\n\n## U.S. news\n\n## U.S. news\n\n## Iranian drone threat prompted U.S. to remove bombers from British base, officials say\n\nA dual British-Iranian national was arrested in London last week on suspicion of plotting a terrorist attack following a security incident at the base.\n\n10d ago\n\n## United Kingdom\n\n## United Kingdom\n\n## Britain believes Iran was involved in incident near air base used by U.S., prime minister says\n\nFive men were arrested on suspicion of terrorist offenses in the incident outside RAF Fairford. Iran had denied any involvement.\n\n11d ago\n\n## United Kingdom\n\n## United Kingdom\n\n## Dual U.K.-Iranian national arrested over security incident at U.S.-run air base in England [...] Prime Minister Andy Bu

In [29]:
print(response["output"])

Recent developments involving Iran include heightened regional tensions and international security concerns. Key reports cover incidents such as security investigations in the U.K. involving a suspected plot and arrests near an air base used by the U.S. (with British officials noting indications of Iranian involvement, which Tehran has denied), concerns over regional conflicts affecting shipping and energy markets, actions involving allied or proxy groups in the Middle East, and ongoing diplomatic discussions regarding U.S.-Iran relations and regional security.
